# E6 Knowledge-Enhanced BioBART

## Objective

## Imports and Setup

In [ ]:
from __future__ import annotations

import gc
import os
import random
import re
from collections import Counter
from pathlib import Path
from typing import Any

LOCAL_CACHE_DIR = Path.cwd() / ".cache"
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(LOCAL_CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(LOCAL_CACHE_DIR))
os.environ.setdefault("HF_HOME", str(LOCAL_CACHE_DIR / "huggingface"))
                                                                 
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import numpy as np
import pandas as pd
import sacrebleu
import torch
from bert_score import score as bert_score
from datasets import Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

try:
    import evaluate
except ImportError:
    evaluate = None

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())



## Configuration

In [ ]:
SEED = 42
MODEL_NAME = "GanjinZero/biobart-v2-base"
MODEL_CANDIDATES = [
    MODEL_NAME,
    "GanjinZero/biobart-base",
    "facebook/bart-base",
]

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "sentence_no_context"
TRAIN_PATH = DATA_DIR / "train_clean.csv"
VAL_PATH = DATA_DIR / "val_clean.csv"
TEST_PATH = DATA_DIR / "test_clean.csv"

KG_DIR = PROJECT_ROOT / "data" / "knowledge_graph"
NODES_PATH = KG_DIR / "nodes.csv"
EDGES_PATH = KG_DIR / "edges.csv"

OUTPUT_DIR = PROJECT_ROOT / "models" / "biobart_knowledge_enhanced"
RESULTS_DIR = PROJECT_ROOT / "results"
PREDICTION_PATH = RESULTS_DIR / "biobart_knowledge_enhanced_predictions.csv"
BEST_MODEL_DIR = OUTPUT_DIR / "best_model"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bf16_enabled = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
fp16_enabled = False

max_source_length = 384
max_target_length = 128

print(f"Project root: {PROJECT_ROOT}")
print(f"Preferred model: {MODEL_NAME}")
print(f"Device: {device}")
print(f"fp16 enabled: {fp16_enabled}")
print(f"bf16 enabled: {bf16_enabled}")
print(f"Prediction path: {PREDICTION_PATH.relative_to(PROJECT_ROOT)}")

## Load Data and Induced Graph

In [ ]:
REQUIRED_DATA_COLUMNS = ["pair_id", "sent_id", "label", "complex", "simple"]

def load_clean_split(path: Path, split_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing {split_name} file: {path}")
    df = pd.read_csv(path)
    missing_columns = [column for column in REQUIRED_DATA_COLUMNS if column not in df.columns]
    if missing_columns:
        raise ValueError(f"{split_name} is missing columns: {missing_columns}")
    df = df[REQUIRED_DATA_COLUMNS].copy()
    for column in ["complex", "simple"]:
        df[column] = df[column].fillna("").astype(str).str.strip()
    df = df[df["complex"].ne("") & df["simple"].ne("")].reset_index(drop=True)
    return df

train_df = load_clean_split(TRAIN_PATH, "train")
val_df = load_clean_split(VAL_PATH, "validation")
test_df = load_clean_split(TEST_PATH, "test")

if not NODES_PATH.exists() or not EDGES_PATH.exists():
    raise FileNotFoundError("Missing induced graph files. Run 06_biomedical_knowledge_graph.ipynb first.")

nodes_df = pd.read_csv(NODES_PATH)
edges_df = pd.read_csv(EDGES_PATH)

print(f"Loaded train: {len(train_df):,}")
print(f"Loaded validation: {len(val_df):,}")
print(f"Loaded test: {len(test_df):,}")
print(f"Graph nodes: {len(nodes_df):,}")
print(f"Graph edges: {len(edges_df):,}")
display(edges_df.head())

## Build Term Simplification Dictionary

In [ ]:
def normalize_term(text: Any) -> str:
    text = str(text).lower().replace("_", " ").strip()
    text = re.sub(r"[^a-z0-9\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def singularize_token(token: str) -> str:
    if len(token) > 4 and token.endswith("ies"):
        return token[:-3] + "y"
    if len(token) > 3 and token.endswith("s") and not token.endswith("ss"):
        return token[:-1]
    return token


def singularize_phrase(phrase: str) -> str:
    return " ".join(singularize_token(token) for token in normalize_term(phrase).split())


def resolve_graph_term(value: Any, nodes: pd.DataFrame) -> str:
    value_str = str(value)
    if "term" in nodes.columns and "node_id" in nodes.columns:
        match = nodes[nodes["node_id"].astype(str).eq(value_str)]
        if len(match):
            return str(match.iloc[0]["term"])
    if "name" in nodes.columns and "node_id" in nodes.columns:
        match = nodes[nodes["node_id"].astype(str).eq(value_str)]
        if len(match):
            return str(match.iloc[0]["name"])
    return value_str


BROAD_SINGLE_WORD_SOURCES = {
    "risk", "adverse", "prevention", "effect", "quality", "rate", "delivery",
}

NOISY_TARGET_PATTERNS = [
    "limited number",
    "large loop excision",
    "debilitating side",
    "the number",
    "there",
]


def usable_mapping_row(row: pd.Series) -> bool:
    source = row["source_norm"]
    target = row["target_norm"]
    source_tokens = source.split()
    if len(source_tokens) == 1 and source in BROAD_SINGLE_WORD_SOURCES:
        return False
    if any(pattern in target for pattern in NOISY_TARGET_PATTERNS):
        return False
    if source == target:
        return False
    return True


def build_term_dictionary(edges: pd.DataFrame, nodes: pd.DataFrame) -> dict[str, str]:
    if "relation" in edges.columns:
        simplified_edges = edges[edges["relation"].fillna("").astype(str).eq("simplified_as")].copy()
    else:
        simplified_edges = edges.copy()

    source_column = "source" if "source" in simplified_edges.columns else "source_id"
    target_column = "target" if "target" in simplified_edges.columns else "target_id"
    if source_column not in simplified_edges.columns or target_column not in simplified_edges.columns:
        raise ValueError("edges.csv must contain source/target or source_id/target_id columns")

    simplified_edges["source_term"] = simplified_edges[source_column].map(lambda value: resolve_graph_term(value, nodes))
    simplified_edges["target_term"] = simplified_edges[target_column].map(lambda value: resolve_graph_term(value, nodes))
    simplified_edges["source_norm"] = simplified_edges["source_term"].map(singularize_phrase)
    simplified_edges["target_norm"] = simplified_edges["target_term"].map(normalize_term)

    if "frequency" not in simplified_edges.columns:
        simplified_edges["frequency"] = 1
    if "supporting_sentence_pairs" not in simplified_edges.columns:
        simplified_edges["supporting_sentence_pairs"] = simplified_edges["frequency"]

    simplified_edges = simplified_edges[simplified_edges.apply(usable_mapping_row, axis=1)].copy()

    simplified_edges = simplified_edges.sort_values(
        ["source_norm", "frequency", "supporting_sentence_pairs"],
        ascending=[True, False, False],
    )
    best_edges = simplified_edges.drop_duplicates(subset=["source_norm"], keep="first")
    return dict(zip(best_edges["source_norm"], best_edges["target_norm"], strict=True))

term_to_simple = build_term_dictionary(edges_df, nodes_df)

print(f"Usable term simplifications: {len(term_to_simple):,}")
display(pd.DataFrame([{"term": k, "simplification": v} for k, v in list(term_to_simple.items())[:30]]))

## Term Detection and Prompt Builder

In [ ]:
def plural_variants(term: str) -> set[str]:
    normalized = normalize_term(term)
    singular = singularize_phrase(normalized)
    variants = {normalized, singular}
    tokens = singular.split()
    if tokens:
        last = tokens[-1]
        plural_last = last[:-1] + "ies" if last.endswith("y") else last + "s"
        variants.add(" ".join(tokens[:-1] + [plural_last]))
    return {variant for variant in variants if variant}


def match_surface_form(sentence_norm: str, term: str) -> str | None:
    variants = sorted(plural_variants(term), key=len, reverse=True)
    for variant in variants:
        pattern = rf"(?<![a-z0-9]){re.escape(variant)}(?![a-z0-9])"
        match = re.search(pattern, sentence_norm)
        if match:
            return match.group(0)
    return None


def find_graph_terms(sentence: str, mapping: dict[str, str], max_terms: int = 5) -> list[tuple[str, str]]:
    sentence_norm = normalize_term(sentence)
    matches = []
    occupied_spans: list[tuple[int, int]] = []

    for term, simple_term in sorted(mapping.items(), key=lambda item: len(item[0]), reverse=True):
        variants = sorted(plural_variants(term), key=len, reverse=True)
        for variant in variants:
            pattern = rf"(?<![a-z0-9]){re.escape(variant)}(?![a-z0-9])"
            match = re.search(pattern, sentence_norm)
            if not match:
                continue
            span = match.span()
            overlaps = any(not (span[1] <= old_start or span[0] >= old_end) for old_start, old_end in occupied_spans)
            if overlaps:
                continue
            matched_term = match.group(0)
            output_simple = simple_term
            if matched_term.endswith("s") and not output_simple.endswith("s") and len(output_simple.split()) <= 3:
                output_simple = output_simple + "s"
            matches.append((matched_term, output_simple))
            occupied_spans.append(span)
            break
        if len(matches) >= max_terms:
            break
    return matches


PROMPT_TEMPLATE = """You are an expert in biomedical text simplification.

Rewrite the biomedical sentence for a general audience.

Rules:
- Preserve the original meaning.
- Use clear and simple language.
- Replace medical, scientific, or technical terms with simpler alternatives whenever possible.
- Remove unnecessary statistical details unless they are essential for understanding the main finding.
- Do not add information that is not present in the original sentence.
- Output exactly one simplified sentence.

Sentence:
{complex_sentence}

Simplified sentence:"""


def build_prompt(complex_sentence: str) -> str:
    return PROMPT_TEMPLATE.format(complex_sentence=str(complex_sentence).strip())


def build_knowledge_prompt(sentence: str, term_pairs: list[tuple[str, str]]) -> str:
    sentence = str(sentence).strip()
    if not term_pairs:
        return build_prompt(sentence)

    mapping_lines = "\n".join(f"- {source} = {target}" for source, target in term_pairs[:globals().get("MAX_TERMS", 5)])
    return f"""You are an expert in biomedical text simplification.

Rewrite the biomedical sentence for a general audience.

Rules:
- Preserve the original meaning.
- Use clear and simple language.
- Replace medical, scientific, or technical terms with simpler alternatives whenever possible.
- Remove unnecessary statistical details unless they are essential for understanding the main finding.
- Do not add information that is not present in the original sentence.
- Output exactly one simplified sentence.

Use these term simplifications if relevant:
{mapping_lines}

Sentence:
{sentence}

Simplified sentence:"""

example_sentence = "Mortality and adverse events were reported."
example_terms = find_graph_terms(example_sentence, term_to_simple)
print(example_terms)
print(build_knowledge_prompt(example_sentence, example_terms))



## Add Knowledge Inputs

In [ ]:
def add_knowledge_columns(df: pd.DataFrame) -> pd.DataFrame:
    enriched = df.copy()
    detected_pairs = enriched["complex"].map(lambda sentence: find_graph_terms(sentence, term_to_simple))
    enriched["detected_terms"] = detected_pairs.map(lambda pairs: "; ".join(f"{source}->{target}" for source, target in pairs))
    enriched["detected_terms_count"] = detected_pairs.map(len)
    enriched["knowledge_input"] = [
        build_knowledge_prompt(sentence, pairs)
        for sentence, pairs in zip(enriched["complex"].tolist(), detected_pairs.tolist(), strict=True)
    ]
    return enriched

train_enriched_df = add_knowledge_columns(train_df)
val_enriched_df = add_knowledge_columns(val_df)
test_enriched_df = add_knowledge_columns(test_df)

for split_name, df in [("train", train_enriched_df), ("validation", val_enriched_df), ("test", test_enriched_df)]:
    coverage = 100 * df["detected_terms_count"].gt(0).mean()
    avg_terms = df["detected_terms_count"].mean()
    print(f"{split_name}: {coverage:.2f}% examples with KG terms; average terms = {avg_terms:.3f}")

all_detected_terms = []
for value in train_enriched_df["detected_terms"]:
    if not isinstance(value, str) or not value:
        continue
    all_detected_terms.extend(value.split("; "))

top_detected_terms_df = pd.DataFrame(Counter(all_detected_terms).most_common(30), columns=["term_mapping", "count"])
display(top_detected_terms_df)

test_coverage = test_enriched_df["detected_terms_count"].gt(0).mean()
if test_coverage < 0.10:
    print("WARNING: The induced graph has low test coverage, so improvements may be limited.")

display(train_enriched_df[["complex", "detected_terms", "knowledge_input"]].head())

## Tokenization

In [ ]:
def load_tokenizer(model_candidates: list[str]) -> tuple[Any, str]:
    errors = []
    for candidate in model_candidates:
        try:
            os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
            tokenizer = AutoTokenizer.from_pretrained(candidate)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            print(f"Loaded tokenizer: {candidate}")
            return tokenizer, candidate
        except Exception as exc:
            errors.append(f"{candidate}: {type(exc).__name__}: {exc}")
            print(f"Could not load tokenizer {candidate}: {exc}")
    raise RuntimeError("Could not load any tokenizer candidate.\n" + "\n".join(errors))

tokenizer, RESOLVED_MODEL_NAME = load_tokenizer(MODEL_CANDIDATES)

def preprocess_examples(examples: dict[str, list[Any]]) -> dict[str, Any]:
    model_inputs = tokenizer(
        examples["knowledge_input"],
        max_length=max_source_length,
        truncation=True,
    )
    labels = tokenizer(
        text_target=examples["simple"],
        max_length=max_target_length,
        truncation=True,
    )["input_ids"]
    labels = [
        [(token_id if token_id != tokenizer.pad_token_id else -100) for token_id in label]
        for label in labels
    ]
    model_inputs["labels"] = labels
    return model_inputs

print(train_enriched_df.loc[0, "knowledge_input"][:1000])


## Dataset Creation

In [ ]:
def make_hf_dataset(df: pd.DataFrame) -> Dataset:
    dataset = Dataset.from_pandas(df, preserve_index=False)
    remove_columns = dataset.column_names
    dataset = dataset.map(preprocess_examples, batched=True, remove_columns=remove_columns)
    return dataset

train_dataset = make_hf_dataset(train_enriched_df)
val_dataset = make_hf_dataset(val_enriched_df)
test_dataset = make_hf_dataset(test_enriched_df)

print(train_dataset)
print(val_dataset)
print(test_dataset)

## Model Loading

In [ ]:
CRITICAL_MISSING_KEYS = {
    "model.encoder.embed_tokens.weight",
    "model.decoder.embed_tokens.weight",
    "lm_head.weight",
}


def load_seq2seq_model(model_candidates: list[str], resolved_tokenizer_model: str) -> tuple[Any, str]:
    ordered_candidates = [resolved_tokenizer_model] + [candidate for candidate in model_candidates if candidate != resolved_tokenizer_model]
    errors = []
    for candidate in ordered_candidates:
        try:
            model, loading_info = AutoModelForSeq2SeqLM.from_pretrained(candidate, output_loading_info=True)
            missing_keys = set(loading_info.get("missing_keys", []))
            critical_missing = sorted(missing_keys & CRITICAL_MISSING_KEYS)
            if critical_missing:
                del model
                gc.collect()
                message = f"critical missing keys: {critical_missing}"
                errors.append(f"{candidate}: {message}")
                print(f"Skipping model {candidate}: {message}")
                continue
            print(f"Loaded model: {candidate}")
            return model, candidate
        except Exception as exc:
            errors.append(f"{candidate}: {type(exc).__name__}: {exc}")
            print(f"Could not load model {candidate}: {exc}")
    raise RuntimeError("Could not load any model candidate without critical missing keys.\n" + "\n".join(errors))

model, RESOLVED_MODEL_NAME = load_seq2seq_model(MODEL_CANDIDATES, RESOLVED_MODEL_NAME)
model.to(device)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

print(f"Resolved model: {RESOLVED_MODEL_NAME}")
print(f"Model device: {next(model.parameters()).device}")

## Training

In [ ]:
def build_training_args() -> Seq2SeqTrainingArguments:
    base_kwargs = dict(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=5,
        learning_rate=3e-5,
        weight_decay=0.01,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=1,
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        predict_with_generate=True,
        fp16=fp16_enabled,
        bf16=bf16_enabled,
        logging_nan_inf_filter=False,
        report_to="none",
        seed=SEED,
    )
    try:
        return Seq2SeqTrainingArguments(evaluation_strategy="epoch", **base_kwargs)
    except TypeError:
        return Seq2SeqTrainingArguments(eval_strategy="epoch", **base_kwargs)

training_args = build_training_args()

def build_trainer() -> Seq2SeqTrainer:
    trainer_kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    try:
        return Seq2SeqTrainer(processing_class=tokenizer, **trainer_kwargs)
    except TypeError:
        return Seq2SeqTrainer(tokenizer=tokenizer, **trainer_kwargs)

trainer = build_trainer()
trainer.train()
trainer.save_model(str(BEST_MODEL_DIR))
tokenizer.save_pretrained(str(BEST_MODEL_DIR))
print(f"Saved best model to: {BEST_MODEL_DIR.relative_to(PROJECT_ROOT)}")

## Training Diagnostics

In [ ]:
training_log_df = pd.DataFrame(trainer.state.log_history)
display(training_log_df)

epoch_eval_df = training_log_df[training_log_df["eval_loss"].notna()].copy()
if len(epoch_eval_df):
    display(epoch_eval_df[["epoch", "step", "eval_loss", "eval_runtime"]])

## Inference

In [ ]:
GENERATION_CONFIG = {
    "max_new_tokens": 128,
    "num_beams": 4,
    "length_penalty": 0.9,
    "no_repeat_ngram_size": 3,
    "early_stopping": True,
}

def clean_prediction(text: str) -> str:
    """Remove prompt echoes and generation boilerplate from decoded text."""
    text = re.sub(r"\s+", " ", str(text).strip())
    if not text:
        return ""

                                                                                   
    prompt_markers = [
        "Simplified sentence:",
        "Rewrite the biomedical sentence for a general audience.",
        "Rewrite this biomedical sentence in simpler language:",
        "Simplify the biomedical sentence.",
        "Sentence:",
    ]
    for marker in prompt_markers:
        if marker in text:
            text = text.split(marker)[-1].strip()

    prefixes = ["Simplified:", "Answer:", "Prediction:"]
    for prefix in prefixes:
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()

    return re.sub(r"\s+", " ", text).strip()

def generate_batch(prompts: list[str]) -> list[str]:
    inputs = tokenizer(
        prompts,
        max_length=max_source_length,
        truncation=True,
        padding=True,
        return_tensors="pt",
    ).to(device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, **GENERATION_CONFIG)
    return [clean_prediction(text) for text in tokenizer.batch_decode(output_ids, skip_special_tokens=True)]


def generate_predictions(df: pd.DataFrame, batch_size: int = 8) -> pd.DataFrame:
    model.eval()
    predictions = []
    prompts = df["knowledge_input"].tolist()
    for start in tqdm(range(0, len(prompts), batch_size), desc="Generating"):
        batch_prompts = prompts[start : start + batch_size]
        try:
            predictions.extend(generate_batch(batch_prompts))
        except Exception as exc:
            print(f"Generation failed for rows {start}-{start + len(batch_prompts) - 1}: {exc}")
            predictions.extend([""] * len(batch_prompts))
    output_df = df[["pair_id", "sent_id", "label", "complex", "simple", "detected_terms", "knowledge_input"]].copy()
    output_df["prediction"] = predictions
    return output_df

prediction_df = generate_predictions(test_enriched_df, batch_size=8)
prediction_df.to_csv(PREDICTION_PATH, index=False)
print(f"Saved predictions to: {PREDICTION_PATH.relative_to(PROJECT_ROOT)}")
display(prediction_df.head())



## Evaluation

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import compute_metrics

metrics_summary = compute_metrics(prediction_df)
display(metrics_summary)


## Ablation Analysis

In [ ]:
def metric_value(summary: pd.DataFrame, metric: str) -> float:
    values = summary.loc[summary["metric"].eq(metric), "score"].tolist()
    return float(values[0]) if values else float("nan")

subsets = {
    "all_test": prediction_df,
    "with_kg_terms": prediction_df[prediction_df["detected_terms"].fillna("").astype(str).str.strip().ne("")].copy(),
    "without_kg_terms": prediction_df[prediction_df["detected_terms"].fillna("").astype(str).str.strip().eq("")].copy(),
}

ablation_rows = []
for subset_name, subset_df in subsets.items():
    if len(subset_df) == 0:
        ablation_rows.append({"subset": subset_name, "rows": 0, "SARI": np.nan, "BLEU": np.nan, "BERTScore F1": np.nan})
        continue
    subset_metrics = compute_metrics(subset_df)
    ablation_rows.append({
        "subset": subset_name,
        "rows": len(subset_df),
        "SARI": metric_value(subset_metrics, "SARI"),
        "BLEU": metric_value(subset_metrics, "BLEU"),
        "BERTScore F1": metric_value(subset_metrics, "BERTScore F1"),
    })

ablation_df = pd.DataFrame(ablation_rows)
display(ablation_df)

## Comparison With Previous Experiments

In [ ]:
comparison_df = pd.DataFrame([
    {"Experiment": "E0", "Model": "Llama 3.1 zero-shot", "SARI": 28.25, "BLEU": 3.60, "BERTScore F1": 0.897},
    {"Experiment": "E1", "Model": "FLAN-T5 fine-tuned", "SARI": 26.86, "BLEU": 36.57, "BERTScore F1": 0.935},
    {"Experiment": "E2", "Model": "BioBART direct", "SARI": 31.86, "BLEU": 30.91, "BERTScore F1": 0.932},
    {"Experiment": "E4", "Model": "Classifier + BioBART", "SARI": 27.01, "BLEU": 22.20, "BERTScore F1": 0.932},
    {
        "Experiment": "E6",
        "Model": f"Knowledge-enhanced BioBART ({RESOLVED_MODEL_NAME})",
        "SARI": metric_value(metrics_summary, "SARI"),
        "BLEU": metric_value(metrics_summary, "BLEU"),
        "BERTScore F1": metric_value(metrics_summary, "BERTScore F1"),
    },
])
display(comparison_df)

if metric_value(metrics_summary, "SARI") < 31.86:
    print("SARI decreased compared with direct BioBART. Possible reasons: noisy extracted mappings, longer prompts, irrelevant graph terms, or BioBART already learned these mappings during fine-tuning.")

## Qualitative Analysis

In [ ]:
with_terms_df = prediction_df[prediction_df["detected_terms"].fillna("").astype(str).str.strip().ne("")].copy()
without_terms_df = prediction_df[prediction_df["detected_terms"].fillna("").astype(str).str.strip().eq("")].copy()

print("Examples with detected graph terms")
if len(with_terms_df):
    display(with_terms_df[["complex", "detected_terms", "simple", "prediction"]].sample(n=min(20, len(with_terms_df)), random_state=SEED))
else:
    print("No test examples contained graph terms.")

print("Examples without detected graph terms")
if len(without_terms_df):
    display(without_terms_df[["complex", "detected_terms", "simple", "prediction"]].sample(n=min(20, len(without_terms_df)), random_state=SEED))
else:
    print("All test examples contained graph terms.")